In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql.functions import col, to_date, count
from pyspark.sql import functions as F

In [ ]:
spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()

df = spark.read.option("header", "true").parquet("yellow_tripdata_2024-10.parquet")

df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

## 1. Size avg partition 

In [12]:
df = df.repartition(4)

In [34]:
type(df)

pyspark.sql.dataframe.DataFrame

In [ ]:
df.repartition(4).write.mode("overwrite").parquet(
    "yellow_tripdata_2024-10-repartitioned.parquet"
)

In [18]:
import os

parquet_dir = "yellow_tripdata_2024-10-repartitioned.parquet/"

# Get sizes of all .parquet files
file_sizes = [
    os.path.getsize(os.path.join(parquet_dir, file))
    / (1024 * 1024)  # Convert bytes to MB
    for file in os.listdir(parquet_dir)
    if file.endswith(".parquet")
]

# Calculate average size
average_size = sum(file_sizes) / len(file_sizes)

print(f"Average Parquet file size: {average_size:.2f} MB")

Average Parquet file size: 23.04 MB


## 2. Count Records

In [ ]:
df = spark.read.option("header", "true").parquet("yellow_tripdata_2024-10.parquet")

In [70]:
df = df.withColumn("pickup_date", to_date(df.tpep_pickup_datetime)).withColumn(
    "dropoff_date", to_date(df.tpep_dropoff_datetime)
)

In [71]:
df.head(10)

[Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2024, 10, 1, 0, 30, 44), tpep_dropoff_datetime=datetime.datetime(2024, 10, 1, 0, 48, 26), passenger_count=1, trip_distance=3.0, RatecodeID=1, store_and_fwd_flag='N', PULocationID=162, DOLocationID=246, payment_type=1, fare_amount=18.4, extra=1.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=24.9, congestion_surcharge=2.5, Airport_fee=0.0, pickup_date=datetime.date(2024, 10, 1), dropoff_date=datetime.date(2024, 10, 1)),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 10, 1, 0, 12, 20), tpep_dropoff_datetime=datetime.datetime(2024, 10, 1, 0, 25, 25), passenger_count=1, trip_distance=2.2, RatecodeID=1, store_and_fwd_flag='N', PULocationID=48, DOLocationID=236, payment_type=1, fare_amount=14.2, extra=3.5, mta_tax=0.5, tip_amount=3.8, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=23.0, congestion_surcharge=2.5, Airport_fee=0.0, pickup_date=datetime.date(2024, 10, 1), d

In [72]:
trip_count = (
    df.filter(col("pickup_date") == "2024-10-15")
    .count()
)

print(f"Number of trips on October 15th: {trip_count}")

Number of trips on October 15th: 128893


## 3. Longest trip

In [74]:
df = df.withColumn(
    "tpep_pickup_datetime", F.col("tpep_pickup_datetime").cast("timestamp")
)
df = df.withColumn(
    "tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp")
)

df = df.withColumn(
    "trip_duration_hours",
    (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    )
    / 3600,
)

max_duration = df.select(F.max("trip_duration_hours")).collect()[0][0]

print(f"longest trip: {max_duration}")

longest trip: 162.61777777777777


## 4. Least frequent pickup location zone

In [75]:
zones_df = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")
zones_df.createOrReplaceTempView("zones")

In [76]:
pickup_counts = df.groupBy("PULocationID").agg(count("*").alias("pickup_count"))

result_df = (
    pickup_counts.join(
        zones_df, pickup_counts.PULocationID == zones_df.LocationID, "left"
    )
    .select(col("Zone"), col("pickup_count"))
    .orderBy("pickup_count")
)

result_df.show(1)

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
+--------------------+------------+
only showing top 1 row

